# Tutorial de PyTorch: De cero al entrenamiento de modelos

Este tutorial te lleva desde los conceptos más básicos hasta entrenar tu primera red neuronal.

**Índice:**
1. Tensores: el bloque fundamental
2. Operaciones con tensores
3. Autograd: cómo PyTorch calcula gradientes
4. Construcción de una red neuronal (`nn.Module`)
5. El bucle de entrenamiento
6. Ejemplo completo: clasificación de dígitos (MNIST)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

print(f'Versión de PyTorch: {torch.__version__}')
print(f'GPU disponible: {torch.cuda.is_available()}')

---
## 1. Tensores: el bloque fundamental

Un **tensor** es un array multidimensional, igual que los arrays de NumPy, pero con superpoders:
- Puede vivir en la GPU
- Registra operaciones para calcular gradientes automáticamente

| Dimensiones | Nombre    | Ejemplo                    |
|-------------|-----------|----------------------------|
| 0D          | Escalar   | `tensor(3.14)`             |
| 1D          | Vector    | `tensor([1, 2, 3])`        |
| 2D          | Matriz    | `tensor([[1,2],[3,4]])`    |
| 3D+         | Tensor    | Imagen, batch de datos...  |

In [ ]:
# Crear tensores de distintas formas
escalar = torch.tensor(3.14)
vector  = torch.tensor([1.0, 2.0, 3.0])
matriz  = torch.tensor([[1.0, 2.0], [3.0, 4.0]])

print('Escalar:', escalar, '| shape:', escalar.shape)
print('Vector: ', vector,  '| shape:', vector.shape)
print('Matriz:\n', matriz,  '| shape:', matriz.shape)

In [ ]:
# Tensores especiales
ceros    = torch.zeros(3, 4)      # 3 filas, 4 columnas de ceros
unos     = torch.ones(2, 3)
aleatorio = torch.rand(3, 3)      # valores entre 0 y 1
arange   = torch.arange(0, 10, 2) # como range() de Python

print('Ceros:\n', ceros)
print('Aleatorio:\n', aleatorio)
print('Arange:', arange)

In [ ]:
# Propiedades importantes de un tensor
t = torch.rand(4, 3)
print('Shape:  ', t.shape)    # dimensiones
print('Dtype:  ', t.dtype)    # tipo de dato (float32 por defecto)
print('Device: ', t.device)   # cpu o cuda

---
## 2. Operaciones con tensores

Las operaciones en PyTorch son similares a NumPy. Lo más importante: las operaciones respetan el **shape** y pueden hacerse elemento a elemento o como álgebra matricial.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print('Suma:        ', a + b)
print('Producto elem:', a * b)   # NO es producto escalar
print('Producto punt:', torch.dot(a, b))  # dot product = 1*4 + 2*5 + 3*6 = 32

In [ ]:
# Multiplicación matricial
A = torch.rand(3, 4)
B = torch.rand(4, 2)
C = A @ B   # equivalente a torch.matmul(A, B)
print('A shape:', A.shape)
print('B shape:', B.shape)
print('C = A @ B shape:', C.shape)  # (3, 2)

In [ ]:
# Reshape: cambiar dimensiones sin cambiar datos
t = torch.arange(12)          # [0, 1, 2, ..., 11]
print('Original:', t.shape)   # (12,)

t2 = t.reshape(3, 4)
print('Reshape (3,4):\n', t2)

t3 = t.reshape(2, 2, 3)
print('Reshape (2,2,3):\n', t3)

### Broadcasting

PyTorch puede operar tensores de shapes distintos si son compatibles, expandiendo automáticamente las dimensiones menores.

In [ ]:
matriz = torch.ones(3, 3)
fila   = torch.tensor([1.0, 2.0, 3.0])  # shape (3,)

# fila se suma a cada fila de la matriz
resultado = matriz + fila
print(resultado)

---
## 3. Autograd: cómo PyTorch calcula gradientes

**Autograd** es el sistema que hace posible el entrenamiento. PyTorch registra todas las operaciones sobre tensores con `requires_grad=True` y puede calcular las derivadas automáticamente.

Esto es la base del **descenso de gradiente**: saber cuánto cambia la pérdida si cambias cada peso.

In [ ]:
# x es una variable que queremos optimizar
x = torch.tensor(3.0, requires_grad=True)

# Definimos una función: y = x^2 + 2x + 1
y = x**2 + 2*x + 1

print('y =', y)

# Calculamos el gradiente: dy/dx
y.backward()

# dy/dx = 2x + 2 → en x=3 → 2*3+2 = 8
print('Gradiente dy/dx en x=3:', x.grad)  # debería ser 8.0

In [ ]:
# Con vectores: gradiente de la suma de cuadrados
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Pérdida = suma de (x_i^2)
loss = (x**2).sum()
loss.backward()

# d(loss)/d(x_i) = 2*x_i
print('x:         ', x.data)
print('Gradientes:', x.grad)  # [2, 4, 6]

### ¿Por qué importa esto?

En una red neuronal, los **pesos** son los tensores con `requires_grad=True`. La **pérdida** mide cuánto se equivoca el modelo. `backward()` calcula cuánto contribuye cada peso al error, y el optimizador los ajusta en la dirección correcta.

```
forward pass → calcular pérdida → backward() → actualizar pesos → repetir
```

---
## 4. Construcción de una red neuronal (`nn.Module`)

En PyTorch, toda red neuronal hereda de `nn.Module`. Solo necesitas definir:
- `__init__`: las capas que usas
- `forward`: cómo fluyen los datos a través de ellas

In [ ]:
class MiRed(nn.Module):
    def __init__(self):
        super().__init__()
        # Capas lineales (fully connected): entrada → oculta → salida
        self.capa1 = nn.Linear(in_features=2, out_features=4)
        self.capa2 = nn.Linear(in_features=4, out_features=1)
        # Función de activación
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.capa1(x))  # capa1 + activación
        x = self.capa2(x)             # capa de salida (sin activación)
        return x


modelo = MiRed()
print(modelo)

In [ ]:
# Ver los parámetros (pesos) del modelo
for nombre, param in modelo.named_parameters():
    print(f'{nombre:20s} | shape: {param.shape} | requires_grad: {param.requires_grad}')

In [ ]:
# Pasar datos por la red (forward pass)
entrada = torch.tensor([[1.0, 2.0]])  # 1 muestra, 2 características
salida  = modelo(entrada)
print('Entrada shape:', entrada.shape)
print('Salida shape: ', salida.shape)
print('Salida:       ', salida)

### Funciones de activación más comunes

Las activaciones introducen **no-linealidad** en la red, lo que le permite aprender patrones complejos.

In [ ]:
x = torch.linspace(-3, 3, 100)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].plot(x.numpy(), torch.relu(x).numpy())
axes[0].set_title('ReLU\nmax(0, x)')
axes[0].grid(True)

axes[1].plot(x.numpy(), torch.sigmoid(x).numpy())
axes[1].set_title('Sigmoid\n1 / (1 + e^-x)')
axes[1].grid(True)

axes[2].plot(x.numpy(), torch.tanh(x).numpy())
axes[2].set_title('Tanh')
axes[2].grid(True)

plt.tight_layout()
plt.show()

---
## 5. El bucle de entrenamiento

Entrenar un modelo siempre sigue este patrón:

```
for cada epoch:
    for cada batch de datos:
        1. optimizer.zero_grad()   ← limpiar gradientes del paso anterior
        2. prediccion = modelo(x)  ← forward pass
        3. loss = criterio(prediccion, y_real)  ← calcular error
        4. loss.backward()         ← calcular gradientes
        5. optimizer.step()        ← actualizar pesos
```

Vamos a verlo con un ejemplo simple: aprender la función `y = 2x + 1`.

In [ ]:
# ---- Datos sintéticos ----
torch.manual_seed(42)
X = torch.rand(100, 1) * 10        # 100 puntos entre 0 y 10
y = 2 * X + 1 + torch.randn(100, 1) * 0.5  # y = 2x + 1 + ruido

plt.scatter(X.numpy(), y.numpy(), alpha=0.5)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos: y ≈ 2x + 1')
plt.grid(True)
plt.show()

In [ ]:
# ---- Modelo: una sola neurona (regresión lineal) ----
modelo_lineal = nn.Linear(in_features=1, out_features=1)
print('Pesos iniciales (aleatorios):')
print(f'  w = {modelo_lineal.weight.item():.4f}')   # debería converger a 2
print(f'  b = {modelo_lineal.bias.item():.4f}')     # debería converger a 1

In [ ]:
# ---- Función de pérdida y optimizador ----
criterio  = nn.MSELoss()                        # Error cuadrático medio
optimizador = optim.SGD(modelo_lineal.parameters(), lr=0.01)  # Descenso de gradiente

# ---- Bucle de entrenamiento ----
historico_loss = []
EPOCHS = 200

for epoch in range(EPOCHS):
    # 1. Limpiar gradientes
    optimizador.zero_grad()

    # 2. Forward pass
    prediccion = modelo_lineal(X)

    # 3. Calcular pérdida
    loss = criterio(prediccion, y)

    # 4. Backward pass (calcular gradientes)
    loss.backward()

    # 5. Actualizar pesos
    optimizador.step()

    historico_loss.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Loss: {loss.item():.4f}')

print(f'\nPesos aprendidos:')
print(f'  w = {modelo_lineal.weight.item():.4f}  (objetivo: 2.0)')
print(f'  b = {modelo_lineal.bias.item():.4f}  (objetivo: 1.0)')

In [ ]:
# Visualizar la curva de aprendizaje y el ajuste
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Curva de pérdida
ax1.plot(historico_loss)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Curva de aprendizaje')
ax1.grid(True)

# Ajuste del modelo
with torch.no_grad():  # no necesitamos gradientes para predecir
    x_linea = torch.linspace(0, 10, 100).reshape(-1, 1)
    y_pred  = modelo_lineal(x_linea)

ax2.scatter(X.numpy(), y.numpy(), alpha=0.5, label='Datos reales')
ax2.plot(x_linea.numpy(), y_pred.numpy(), 'r-', linewidth=2, label='Modelo aprendido')
ax2.set_xlabel('X')
ax2.set_ylabel('y')
ax2.set_title('Ajuste del modelo')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## 6. Ejemplo completo: clasificación de dígitos (MNIST)

Ahora juntamos todo: dataset real, red más profunda, clasificación multiclase.

MNIST es una colección de imágenes de dígitos escritos a mano (0-9). Cada imagen es de 28×28 píxeles.

In [ ]:
# ---- Descargar y preparar los datos ----
transformacion = transforms.Compose([
    transforms.ToTensor(),             # Convierte PIL Image → tensor [0,1]
    transforms.Normalize((0.5,), (0.5,))  # Normaliza a [-1, 1]
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transformacion)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transformacion)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f'Ejemplos de entrenamiento: {len(train_dataset)}')
print(f'Ejemplos de test:          {len(test_dataset)}')
print(f'Número de batches (train): {len(train_loader)}')

In [ ]:
# Visualizar algunos ejemplos
imagenes, etiquetas = next(iter(train_loader))  # primer batch
print('Shape del batch:', imagenes.shape)  # (64, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imagenes[i].squeeze(), cmap='gray')
    ax.set_title(str(etiquetas[i].item()))
    ax.axis('off')
plt.suptitle('Muestra del dataset MNIST')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Definir la red ----
class ClasificadorMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            nn.Flatten(),              # (batch, 1, 28, 28) → (batch, 784)
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2),           # regularización: apaga el 20% de neuronas al azar
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)         # 10 clases (dígitos 0-9)
        )

    def forward(self, x):
        return self.red(x)


# Usar GPU si está disponible
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Usando: {device}')

modelo_mnist = ClasificadorMNIST().to(device)
print(modelo_mnist)

# Contar parámetros
total_params = sum(p.numel() for p in modelo_mnist.parameters())
print(f'\nTotal de parámetros: {total_params:,}')

In [ ]:
# ---- Configurar entrenamiento ----
# CrossEntropyLoss = Softmax + Log + NLLLoss (ideal para clasificación multiclase)
criterio    = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_mnist.parameters(), lr=0.001)  # Adam adapta el lr automáticamente


def entrenar_epoch(modelo, loader, criterio, optimizador):
    """Entrena el modelo durante una epoch y devuelve la pérdida media."""
    modelo.train()  # modo entrenamiento (activa Dropout, BatchNorm, etc.)
    loss_total = 0.0

    for imagenes, etiquetas in loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        optimizador.zero_grad()
        salida = modelo(imagenes)
        loss   = criterio(salida, etiquetas)
        loss.backward()
        optimizador.step()

        loss_total += loss.item()

    return loss_total / len(loader)


def evaluar(modelo, loader, criterio):
    """Evalúa el modelo en un conjunto de datos y devuelve pérdida y exactitud."""
    modelo.eval()  # modo evaluación (desactiva Dropout)
    loss_total  = 0.0
    correctas   = 0

    with torch.no_grad():  # no necesitamos gradientes al evaluar
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salida = modelo(imagenes)
            loss_total += criterio(salida, etiquetas).item()
            predicciones = salida.argmax(dim=1)  # clase con mayor probabilidad
            correctas    += (predicciones == etiquetas).sum().item()

    loss_media = loss_total / len(loader)
    exactitud  = correctas / len(loader.dataset) * 100
    return loss_media, exactitud

In [ ]:
# ---- Bucle de entrenamiento completo ----
EPOCHS = 5
hist = {'train_loss': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss            = entrenar_epoch(modelo_mnist, train_loader, criterio, optimizador)
    test_loss, test_acc   = evaluar(modelo_mnist, test_loader, criterio)

    hist['train_loss'].append(train_loss)
    hist['test_loss'].append(test_loss)
    hist['test_acc'].append(test_acc)

    print(f'Epoch {epoch}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Test Loss: {test_loss:.4f} | '
          f'Test Acc: {test_acc:.2f}%')

In [ ]:
# Visualizar resultados
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, hist['train_loss'], label='Train')
ax1.plot(epochs_range, hist['test_loss'],  label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Pérdida por epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs_range, hist['test_acc'], color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Exactitud (%)')
ax2.set_title('Exactitud en test')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Ver algunas predicciones
modelo_mnist.eval()
imagenes_test, etiquetas_test = next(iter(test_loader))
imagenes_test = imagenes_test.to(device)

with torch.no_grad():
    salida = modelo_mnist(imagenes_test)
    predicciones = salida.argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(imagenes_test[i].cpu().squeeze(), cmap='gray')
    pred  = predicciones[i].item()
    real  = etiquetas_test[i].item()
    color = 'green' if pred == real else 'red'
    ax.set_title(f'P:{pred} R:{real}', color=color, fontsize=8)
    ax.axis('off')

plt.suptitle('Predicciones (verde=correcto, rojo=error)')
plt.tight_layout()
plt.show()

---
## Guardar y cargar un modelo

No pierdas el trabajo de entrenamiento: guarda los pesos para reutilizarlos.

In [ ]:
# Guardar solo los pesos (recomendado)
torch.save(modelo_mnist.state_dict(), 'modelo_mnist.pth')
print('Modelo guardado como modelo_mnist.pth')

# Cargar los pesos
modelo_cargado = ClasificadorMNIST().to(device)
modelo_cargado.load_state_dict(torch.load('modelo_mnist.pth', map_location=device))
modelo_cargado.eval()
print('Modelo cargado correctamente')

# Verificar que da los mismos resultados
_, acc = evaluar(modelo_cargado, test_loader, criterio)
print(f'Exactitud del modelo cargado: {acc:.2f}%')

---
## Resumen y próximos pasos

### Lo que hemos aprendido:

| Concepto | Qué es | Para qué sirve |
|----------|--------|----------------|
| **Tensor** | Array N-dimensional | Almacenar datos y pesos |
| **Autograd** | Sistema de diferenciación automática | Calcular gradientes |
| **nn.Module** | Clase base para redes | Definir arquitecturas |
| **Loss function** | Mide el error del modelo | Guía el aprendizaje |
| **Optimizer** | Actualiza los pesos | SGD, Adam, etc. |
| **DataLoader** | Carga datos en batches | Entrenar eficientemente |
| **train() / eval()** | Modos del modelo | Controlar Dropout, BatchNorm |

### Próximos pasos sugeridos:

1. **Redes convolucionales (CNN)** — para imágenes, usan `nn.Conv2d`
2. **Transfer Learning** — usar modelos preentrenados con `torchvision.models`
3. **Datasets propios** — hereda de `torch.utils.data.Dataset`
4. **Learning rate scheduling** — ajustar el lr durante el entrenamiento
5. **Batch Normalization** — `nn.BatchNorm2d` para entrenamientos más estables

**Recursos:**
- [Documentación oficial PyTorch](https://pytorch.org/docs/stable/)
- [Tutoriales oficiales](https://pytorch.org/tutorials/)
- [PyTorch en 60 minutos](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html)